In [ ]:
import pandas as pd
import datetime as dt
import numpy as np

from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import plotly.express as px


In [ ]:
import os
if os.path.exists('pass.env'):
    load_dotenv('pass.env')
else:
    load_dotenv('../pass.env')

# Environment Variables
DB_TYPE = os.getenv('DB_TYPE')
DB_USER = os.getenv('DB_USER')
DB_PASS = os.getenv('DB_PASS')
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_NAME')


In [ ]:
# Connect to PostgreSQL engine
connection_string = f"{DB_TYPE}://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string)


In [ ]:
query = """
WITH customer_rfm AS (
    SELECT 
        dc.customer_unique_id,
        MAX(fs.order_date) AS last_order_date,
        COUNT(DISTINCT fs.order_id) AS "Frequency",
        SUM(fs.gross_revenue) AS "Monetary"
    FROM fact_sales fs
    JOIN dim_customer dc ON fs.customer_id = dc.customer_id
    GROUP BY dc.customer_unique_id
),
snapshot AS (
    SELECT MAX(order_date) + INTERVAL '1 day' AS snapshot_date FROM fact_sales
)
SELECT 
    cr.customer_unique_id,
    (s.snapshot_date::date - cr.last_order_date::date) AS "Recency",
    cr."Frequency",
    cr."Monetary"
FROM customer_rfm cr
CROSS JOIN snapshot s;
"""
df = pd.read_sql_query(query, con=engine)


In [ ]:
# Prepare RFM DataFrame
rfm = df.set_index('customer_unique_id')

print("RFM Data Sample:")
print(rfm.head())

print("\nRFM Data Summary:")
print(rfm.describe())


In [ ]:
# 1. Feature Preprocessing (Log Transformation & Scaling)
rfm_log = rfm[['Recency', 'Frequency', 'Monetary']].apply(np.log1p)

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)
rfm_scaled_df = pd.DataFrame(rfm_scaled, columns=['Recency', 'Frequency', 'Monetary'])

# 2. Train K-Means Clustering Model (k=4)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
kmeans.fit(rfm_scaled_df)

rfm['Cluster'] = kmeans.labels_

print("Cluster Summary (Means & Counts):")
cluster_summary = rfm.groupby('Cluster').agg({
    'Recency': 'mean',
    'Frequency': 'mean',
    'Monetary': ['mean', 'count']
}).round(2)
print(cluster_summary)

# 3. Interactive 3D Scatter Plot
rfm_plot = rfm.reset_index().copy()
rfm_plot['Cluster_Label'] = 'Cluster ' + rfm_plot['Cluster'].astype(str)

fig = px.scatter_3d(
    rfm_plot,
    x='Recency',
    y='Frequency',
    z='Monetary',
    color='Cluster_Label',
    opacity=0.7,
    title='3D RFM Customer Segmentation Map',
    labels={
        'Recency': 'Recency (Days)',
        'Frequency': 'Frequency (Orders)',
        'Monetary': 'Monetary (Spend)'
    },
    color_discrete_sequence=px.colors.qualitative.Set1
)
fig.update_traces(marker=dict(size=4, line=dict(width=0)))
fig.show()


In [ ]:
# Map K-Means cluster IDs to business persona names
segment_mapping = {
    0: 'Promising',
    1: 'At Risk',
    2: 'Champions & Loyal',
    3: 'Lost / Hibernating'
}
rfm['rfm_segment'] = rfm['Cluster'].map(segment_mapping)

# Format columns to standard database schema
rfm_export = rfm.reset_index()[['customer_unique_id', 'Recency', 'Frequency', 'Monetary', 'Cluster', 'rfm_segment']]
rfm_export.columns = ['customer_unique_id', 'recency_days', 'frequency_orders', 'monetary_spend', 'cluster_id', 'rfm_segment']

# Export dataset to PostgreSQL table dim_customer_rfm
rfm_export.to_sql('dim_customer_rfm', engine, if_exists='replace', index=False)
print('Successfully exported dim_customer_rfm to PostgreSQL database!')
